In [0]:
from pyspark.sql.functions import *

---

# **Incremental Data Loading**
Incrementally load the data using auto loader using the Directory Listing method. Both Idempotency and Schema evolution are managed by:

- **File Loading Idempotency**: The checkpoint is stored in the `CheckpointLocation` that further has a `rocksdb`, this guarantees Idempotency and ensures only new files are processed everytime.

- **Schema Handling**: When the data is first loaded, the schema is inferred, cached and saved in `SchemaLocation` and if the schema evolved, it will first write the schema in the `SchemaLocation` and then try to match the cached schema and evolve the schema while writing the data in the location

In [0]:
# Get the source and target folder
dbutils.widgets.text("source_folder", "")
dbutils.widgets.text("target_folder", "")

# Get the source and target folder parameters values from above and store in the variable
src_folder = dbutils.widgets.get("source_folder")
tgt_folder = dbutils.widgets.get("target_folder")

base = "abfss://bronze@netflixprojectetedlll.dfs.core.windows.net"
checkpoint_path = f"{base}/checkpoints/{tgt_folder}"
schema_path     = f"{base}/schemas/{tgt_folder}"

In [0]:
# Incrementally Read and load the data from the raw container
df = spark.readStream.format("cloudFiles") \
                     .option("cloudFiles.format", "csv") \
                     .option("cloudFiles.schemaLocation", schema_path) \
                     .option("quote", '"') \
                     .option("escape", '"') \
                     .load(f"abfss://raw@netflixprojectetedlll.dfs.core.windows.net/{src_folder}")

In [0]:
# Add the ingestion date column for the incremental read in Silver
if tgt_folder == "netflix_titles":
    df = df.withColumn("_ingested_date", current_timestamp()) \
           .withColumn("_file_modified", col("_metadata.file_modification_time"))

In [0]:
# Write the data in Bronze layer
df.writeStream.format("delta") \
              .outputMode("append") \
              .option("checkpointLocation", checkpoint_path) \
              .option("mergeSchema", "true") \
              .trigger(availableNow=True) \
              .start(f"abfss://bronze@netflixprojectetedlll.dfs.core.windows.net/{tgt_folder}")

---

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType

ACC = "netflixprojectetedlll.dfs.core.windows.net"
RAW, BRONZE = f"abfss://raw@{ACC}", f"abfss://bronze@{ACC}"
SILVER, GOLD = "netflix_catalog.silver", "netflix_catalog.gold"
KNOWN_ORPHANS = {80119194, 80188902}          # source-file defects
BRIDGES = {"netflix_cast": "cast", "netflix_category": "listed_in",
           "netflix_countries": "country", "netflix_directors": "director"}
GOLD_MAP = {"netflix_category": ("listed_in", "dim_category", "DimCategoryKey", "category", "bridge_show_category"),
            "netflix_directors": ("director", "dim_director", "DimDirectorKey", "director", "bridge_show_director"),
            "netflix_cast": ("cast", "dim_cast", "DimCastKey", "cast_member", "bridge_show_cast"),
            "netflix_countries": ("country", "dim_country", "DimCountryKey", "country", "bridge_show_country")}
results = []

def check(name, ok, detail="", warn=False):
    st = "PASS" if ok else ("WARN" if warn else "FAIL")
    results.append(st); print(f"[{st}] {name}" + (f"  ->  {detail}" if detail else ""))

def section(title):
    def deco(fn):
        print(f"\n===== {title} =====")
        try: fn()
        except Exception as e: check(f"{title}: crashed", False, f"{type(e).__name__}: {str(e)[:200]}")
        return fn
    return deco

def read_raw(folder):
    df = (spark.read.option("header", "true").option("quote", '"').option("escape", '"')
          .csv(f"{RAW}/{folder}"))
    return df.toDF(*[c.replace("\ufeff", "").strip() for c in df.columns])

clean = lambda c: trim(regexp_replace(col(c), r"[,\s]+$", ""))

@section("1. RAW -> BRONZE")
def _():
    for t in ["netflix_titles", *BRIDGES]:
        b = spark.read.format("delta").load(f"{BRONZE}/{t}")
        r, bn = read_raw(t).count(), b.count()
        check(f"bronze.{t} rows = raw rows", r == bn, f"raw={r}, bronze={bn}")
    b = spark.read.format("delta").load(f"{BRONZE}/netflix_titles")
    bad = b.filter(col("show_id").try_cast("int").isNull()).count()
    check("bronze titles: non-numeric show_id (4 = source defect)", bad <= 4, f"{bad}")
    check("bronze titles: _ingested_date present, no NULLs",
          "_ingested_date" in b.columns and b.filter(col("_ingested_date").isNull()).count() == 0)

@section("2. SILVER titles")
def _():
    s = spark.table(f"{SILVER}.netflix_titles"); n = s.count()
    exp = (read_raw("netflix_titles").withColumn("show_id", col("show_id").try_cast("int"))
           .filter(col("show_id").isNotNull() & col("type").isin("Movie", "TV Show"))
           .select("show_id").distinct())
    check("show_id unique", n == s.select("show_id").distinct().count(), f"{n} rows")
    check("show_id is int, no NULLs", isinstance(s.schema["show_id"].dataType, IntegerType)
          and s.filter(col("show_id").isNull()).count() == 0)
    check("silver shows = distinct valid shows in raw", n == exp.count(), f"silver={n}, raw={exp.count()}")
    check("no missing shows", exp.join(s.select("show_id"), "show_id", "left_anti").count() == 0)
    check("no extra shows", s.select("show_id").join(exp, "show_id", "left_anti").count() == 0)
    check("type only Movie / TV Show", s.filter(~col("type").isin("Movie", "TV Show")).count() == 0)
    check("durations no NULLs", s.filter(col("duration_minutes").isNull() | col("duration_seasons").isNull()).count() == 0)
    check("type_flag matches type", s.filter(((col("type") == "Movie") & (col("type_flag") != 1)) |
                                            ((col("type") == "TV Show") & (col("type_flag") != 2))).count() == 0)
    check("short_title correct", s.filter(col("short_title") != split(col("title"), ":")[0]).count() == 0)
    check("Heineken present", s.filter(col("title") == "Kidnapping Mr. Heineken").count() == 1)
    nd = s.filter(col("date_added").isNull() | (trim(col("date_added")) == "")).count()
    check("shows with no date_added", nd == 0, f"{nd}", warn=True)
    bd = s.filter(trim(col("date_added")) != "").filter(to_date(col("date_added"), "M/d/yyyy").isNull()).count()
    check("date_added parses", bd == 0, f"{bd} unparseable", warn=True)

@section("3. SILVER bridges")
def _():
    ids = spark.table(f"{SILVER}.netflix_titles").select("show_id").distinct()
    for t, vc in BRIDGES.items():
        df = spark.table(f"{SILVER}.{t}"); n = df.count()
        check(f"{t}: no NULL/empty values", df.filter(col("show_id").isNull() | col(vc).isNull() | (trim(col(vc)) == "")).count() == 0)
        check(f"{t}: no trailing comma/space", df.filter(col(vc).rlike(r"[,\s]+$") | col(vc).rlike(r"^\s")).count() == 0)
        check(f"{t}: no duplicate pairs", n == df.select("show_id", vc).distinct().count())
        exp = (read_raw(t).withColumn("show_id", col("show_id").try_cast("int")).withColumn(vc, clean(vc))
               .filter(col("show_id").isNotNull() & col(vc).isNotNull() & (col(vc) != ""))
               .select("show_id", vc).distinct())
        sv = df.select("show_id", vc)
        m, x = exp.join(sv, ["show_id", vc], "left_anti").count(), sv.join(exp, ["show_id", vc], "left_anti").count()
        check(f"{t}: matches raw", m == 0 and x == 0, f"missing={m}, extra={x}")
        orph = {r[0] for r in df.select("show_id").distinct().join(ids, "show_id", "left_anti").collect()}
        check(f"{t}: orphan show_ids", orph <= KNOWN_ORPHANS, f"{sorted(orph)}", warn=orph <= KNOWN_ORPHANS)

@section("4. GOLD")
def _():
    g = lambda n: spark.table(f"{GOLD}.{n}")
    st = spark.table(f"{SILVER}.netflix_titles"); ids = st.select("show_id"); nt = st.count()
    ds, dd, f = g("dim_show"), g("dim_date"), g("fact_show")
    for t, (sc, dim, key, biz, br) in GOLD_MAP.items():
        s, d, b = spark.table(f"{SILVER}.{t}"), g(dim), g(br)
        check(f"{dim}: key unique, no NULLs", d.count() == d.select(key).distinct().count() and d.filter(col(key).isNull()).count() == 0)
        check(f"{dim}: rows = distinct silver values", d.count() == s.select(trim(col(sc))).distinct().count())
        nb = b.count()
        e_all = s.select("show_id", trim(col(sc))).distinct().count()
        e_semi = s.join(ids, "show_id", "left_semi").select("show_id", trim(col(sc))).distinct().count()
        check(f"{br}: row count", nb in (e_all, e_semi), f"gold={nb}, silver all={e_all}, known shows only={e_semi}")
        check(f"{br}: no duplicate rows", nb == b.distinct().count())
        check(f"{br}: every {key} in {dim}", b.join(d, key, "left_anti").count() == 0)
        orph = {r[0] for r in b.select("show_id").distinct().join(ds.select("show_id"), "show_id", "left_anti").collect()}
        check(f"{br}: show_ids in dim_show", not orph, f"orphans={sorted(orph)}" if orph else "", warn=bool(orph) and orph <= KNOWN_ORPHANS)
        nl = ds.select("show_id").join(b.select("show_id").distinct(), "show_id", "left_anti").count()
        check(f"shows with no rows in {br}", nl == 0, f"{nl}", warn=True)
    check("dim_show = silver titles", ds.count() == nt and ds.select("show_id").distinct().count() == nt, f"{ds.count()}")
    check("fact_show = silver titles, 1 row per show", f.count() == nt and f.select("show_id").distinct().count() == nt)
    check("dim_date key unique and = yyyyMMdd", dd.count() == dd.select("DimDateKey").distinct().count()
          and dd.filter(col("DimDateKey") != date_format(col("full_date"), "yyyyMMdd").cast("int")).count() == 0)
    check("fact DimDateKeys exist in dim_date",
          f.filter(col("DimDateKey").isNotNull()).join(dd.select("DimDateKey"), "DimDateKey", "left_anti").count() == 0)
    check("fact -> dim_show/dim_date no fan-out",
          f.join(ds, "show_id", "left").count() == f.count() and f.join(dd, "DimDateKey", "left").count() == f.count())
    check("fact: no zero durations", f.filter((col("duration_minutes") == 0) | (col("duration_seasons") == 0)).count() == 0)

print(f"\n{results.count('PASS')} passed | {results.count('WARN')} warnings | {results.count('FAIL')} failed")


===== 1. RAW -> BRONZE =====
[PASS] bronze.netflix_titles rows = raw rows  ->  raw=6236, bronze=6236
[PASS] bronze.netflix_cast rows = raw rows  ->  raw=44311, bronze=44311
[PASS] bronze.netflix_category rows = raw rows  ->  raw=13670, bronze=13670
[PASS] bronze.netflix_countries rows = raw rows  ->  raw=7179, bronze=7179
[PASS] bronze.netflix_directors rows = raw rows  ->  raw=4852, bronze=4852
[PASS] bronze titles: non-numeric show_id (4 = source defect)  ->  4
[PASS] bronze titles: _ingested_date present, no NULLs

===== 2. SILVER titles =====
[PASS] show_id unique  ->  6232 rows
[PASS] show_id is int, no NULLs
[PASS] silver shows = distinct valid shows in raw  ->  silver=6232, raw=6232
[PASS] no missing shows
[PASS] no extra shows
[PASS] type only Movie / TV Show
[PASS] durations no NULLs
[PASS] type_flag matches type
[PASS] short_title correct
[PASS] Heineken present
[WARN] shows with no date_added  ->  11
[PASS] date_added parses  ->  0 unparseable

===== 3. SILVER bridges =====